# Predict passage readability with text features

Predict synthetic passage easiness from passage text, title, and word count using Structured Data Models and pretrained TabICLv2.
This Databricks notebook uses CPU by default and requires Python 3.11 or later.
See the [TabICLv2 API](https://nvidia.github.io/structured-data-models/api/generated/sdm.models.TabICLv2) for the prediction interface.

## Setup

Install the latest package and its optional text encoder, then restart Python before importing them.
See the [SDM installation guide](https://nvidia.github.io/structured-data-models/install.html) for package requirements.


In [ ]:
%pip install structured-data-models sentence-transformers


In [ ]:
dbutils.library.restartPython()


<br>

### Parameters

Use Sentence Transformer embeddings followed by PCA to represent text.
The seed controls the random holdout, and eight estimators combine predictions from different preprocessing configurations.
CPU is the default; set `device` to `torch.device("cuda")` on GPU compute.
The output path is a new writable directory on the driver.


In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
# Modified for the consolidated examples: Databricks setup and batch outputs.

import tempfile
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
import torch

import sdm
import sdm.processing as sp

seed = 0
device = torch.device("cpu")
num_estimators = 8
output_path = Path(tempfile.mkdtemp(prefix="synthetic_readability_")) / "predictions.parquet"
torch.manual_seed(seed)


<br>

## Generate synthetic example data

Run the next cell to create synthetic passage IDs, titles, excerpts, word counts, and a noisy target called `BT Easiness`. Higher generated complexity lowers this target; it is a demonstration score rather than a measured corpus label.
You can skim generation; context preparation starts below.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def generate(output, seed=42):
    rng = np.random.default_rng(seed)
    n = 1800
    complexity = rng.uniform(0, 1, n)
    simple = "The small bird rests near the green tree. "
    complex_text = "Ecological interdependence illustrates substantial environmental variability. "
    passages = [((simple * (10 - int(c * 8))) + (complex_text * (1 + int(c * 8)))) for c in complexity]
    frame = pd.DataFrame({"ID": np.arange(n), "Title": [f"Synthetic passage {i}" for i in range(n)], "Excerpt": passages, "Word Count": [len(t.split()) for t in passages], "BT Easiness": (1 - 3 * complexity + rng.normal(0, 0.2, n)).round(4)})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    path = output / "passages.parquet"
    frame.to_parquet(path, index=False, compression=None, use_dictionary=False)
    return path

data_path = generate(output_path.parent / "data", seed=seed)
arrow_table = pq.read_table(data_path)
target_name = "BT Easiness"
display(arrow_table.slice(0, 5).to_pandas())


<br>

## Prepare labeled context

Reserve 80% of shuffled passages as context and hold out the rest for scoring. Features are the generated `Title`, `Excerpt`, and `Word Count`; exclude `ID` and retain it separately for output alignment.
Semantic-type inference uses the full table, while model preprocessing fits on context rows. These synthetic holdout metrics do not describe a text-only or external benchmark.


In [ ]:
table = sdm.TableTensor.from_arrow(
    table=arrow_table.drop(["ID"]),
    stypes=sdm.infer_stypes(arrow_table.drop(["ID"]), text="infer"),
    device=device,
)
row_order = torch.randperm(table.size(0), device=device)
shuffled_ids = arrow_table["ID"].take(row_order.cpu().numpy())
table = table[row_order]
context_rows = int(0.8 * table.size(0))
context, query = table.split(context_rows)
query_ids = shuffled_ids.slice(context_rows)
print(f"Context rows: {context.size(0)}, held-out rows: {query.size(0)}")


<br>

## Configure pretrained regression

TabICLv2 predicts from labeled context without a separate training loop.
Apply the [text processor](https://nvidia.github.io/structured-data-models/api/generated/sdm.processing.text.SentenceTransformer.html) only to text columns, then compress their embeddings to 64 features with PCA fitted on context rows.
The default recipe handles the remaining numerical and categorical features.


In [ ]:
model = sdm.models.TabICLv2(task="regression", device=device)
recipe = model.default_recipe()
recipe.prepend_features(
    sp.StypeDispatch(
        text=[
            sp.SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"),
            sp.PCA(64),
        ],
    )
)


<br>

## Batch predictions

Score every held-out passage using the configured pretrained model and labeled context.
Remove `BT Easiness` from both context and scoring features; only the context labels go to the model.


In [ ]:
x_context = context.drop_columns(target_name)
y_context = context[:, target_name]
x_query = query.drop_columns(target_name)
assert x_query.size(0) == len(query_ids)


Predict the held-out batch in one model call.
The recipe fits preprocessing on context rows and transforms query rows with that state.


In [ ]:
with torch.amp.autocast(device.type, torch.float16, enabled=table.is_cuda):
    quantiles = model(
        x_context=x_context,
        y_context=y_context,
        x_query=x_query,
        recipe=recipe,
        num_estimators=num_estimators,
    ).numerical

# The recipe averages estimators; the remaining axis holds 999 quantiles.
predictions = quantiles.mean(dim=-1)
assert predictions.shape == (x_query.size(0),)


Each input passage produces one row: `ID` is the generated passage identifier, and `predicted_bt_easiness` is the mean of its predicted quantiles.
Verify the row count and ID order, then inspect a sample.


In [ ]:
results = pa.table({
    "ID": query_ids,
    "predicted_bt_easiness": predictions.detach().cpu().numpy(),
})
assert results.num_rows == query.size(0)
assert results["ID"].equals(query_ids)
display(results.slice(0, 10).to_pandas())


Save all batch predictions as Parquet.
The printed driver-local path is temporary; copy the file to retained storage before the compute is terminated if you need it later.


In [ ]:
pq.write_table(results, output_path)
print(f"Saved {results.num_rows} predictions to {output_path}")


<br>

## Evaluate the holdout

**Synthetic-data metrics — demonstration only.** Compare predictions with held-out labels after scoring.
RMSE and MAE are in generated `BT Easiness` units; smaller values mean smaller errors, and RMSE gives larger errors more weight.


In [ ]:
y_query = query[:, target_name].numerical.squeeze(-1)
rmse = (predictions - y_query).pow(2).mean().sqrt()
mae = (predictions - y_query).abs().mean()
print(f"RMSE: {rmse:.3f}, MAE: {mae:.3f} (BT Easiness units)")
